In [1]:

import ipywidgets as widgets
from IPython.display import display
import datetime, json, uuid

# ============================================================
# PRO174 – Mantenimiento Preventivo
# HMI oficial (BASE estilo PRO134/PRO173 aprobado)
#
# Reglas HMI:
# - Títulos de pasos = casillas del flujo (texto exacto)  [excepto selectores HMI]
# - Checklist por paso (no avanza si no se completa)
# - Botón NO => BLOQUEADO con motivo obligatorio + "Rehacer paso"
# - Volver al paso anterior
# - Exporta JSON auditable (run_id, historial, decisiones, bloqueos, timestamps, estados)
# ============================================================

EN_CURSO = "EN_CURSO"
BLOQUEADO = "BLOQUEADO"
DETENIDO_STOP = "DETENIDO_STOP"
FINALIZADO = "FINALIZADO"

def _now_iso():
    return datetime.datetime.now().isoformat(timespec="seconds")

# Motivos de bloqueo (multi-selección) - base operacional
MOTIVOS_BLOQUEO_PRO174 = [
    "PT no vigente / condiciones de PT no cumplen",
    "Condición de seguridad no controlada (energías, accesos, EPP, etc.)",
    "Recursos no disponibles (personal / herramientas / repuestos / servicios)",
    "OT inconsistente (prioridad, alcance, UT/Proceso, operaciones)",
    "Sistema SAP indisponible / sin acceso (IW31 / IW32 / IW41 / IW48)",
    "No se puede ejecutar según planificación (ventana, coordinación, permisos)",
    "Requiere autorización / aprobación no obtenida (servicios externos, etc.)",
    "Otro"
]

# ============================================================
# NODOS (PRO174)
# Nota: Se incluye selector de inicio HMI (no es paso del PRO) para operación real.
# Secuencia base (según flujo + corrección validada): Crear OT -> Preparar OT -> Asignar PEP -> Liberar OT
# ============================================================
NODOS = {

    # ===== Selector de inicio (HMI) =====
    "S0_inicio": {
        "type": "decision",
        "titulo": "¿Desde qué paso iniciar el PRO174?",
        "rol": "HMI (selección de inicio)",
        "descripcion": "Seleccione el punto de entrada operativo según el estado real del trabajo (OT creada / preparación / ejecución).",
        "pregunta": "Seleccione una opción para iniciar:",
        "opciones": [
            {"label": "Inicio 1: Creación de OT del sistema (OT02/OT06/OT8)", "next": "T1_Creacion_OT"},
            {"label": "Inicio 2: Preparar OT", "next": "T2_Preparar_OT"},
            {"label": "Inicio 3: Preparar trabajos", "next": "T5_Preparar_trabajos"},
        ],
        "ayuda": "Selector HMI: no modifica el flujo del PRO, solo define desde dónde comenzar."
    },

    # ---- Troncal OT ----
    "T1_Creacion_OT": {
        "type": "task",
        "titulo": "Creación de OT del sistema (OT02/OT06/OT8)",
        "rol": "Especialización de mantenimiento",
        "descripcion": "Crear orden de trabajo del sistema para mantenimiento preventivo. Transacción: IW31.",
        "acciones": [
            "Crear orden de trabajo del sistema (OT02/OT06/OT8) en SAP.",
            "Confirmar datos mínimos del preventivo (equipo/UT, alcance, fecha/ventana).",
        ],
        "checklist": [
            "OT creada en SAP (IW31)",
            "Datos mínimos del preventivo confirmados (equipo/UT, alcance, fecha/ventana)"
        ],
        "validacion": "¿La OT del sistema quedó creada en SAP (IW31) con los datos mínimos del preventivo?",
        "next": "T2_Preparar_OT"
    },

    "T2_Preparar_OT": {
        "type": "task",
        "titulo": "Preparar OT",
        "rol": "Especialista/ Ingeniero de Mantenimiento",
        "descripcion": (
            "Documentar el requerimiento de mantenimiento.\n"
            "- Definir las tareas de mantenimiento (operaciones).\n"
            "- Definir duración de los trabajos y N° de personas requeridas por tarea.\n"
            "- Definir el Ubicación Técnica por función.\n"
            "- Definir los materiales requeridos.\n"
            "- Incorporar Servicios como Operación (PM03).\n"
            "Transacción: IW32"
        ),
        "acciones": [
            "En IW32, documentar requerimiento y definir operaciones.",
            "Definir duración y dotación por tarea.",
            "Definir Ubicación Técnica por función.",
            "Definir materiales requeridos.",
            "Incorporar servicios como operación (PM03) si aplica."
        ],
        "checklist": [
            "Operaciones definidas en OT",
            "Duración definida",
            "Dotación definida",
            "Ubicación Técnica por función definida",
            "Si aplica: Materiales requeridos definidos",
            "Si aplica: servicios incorporados como operación (PM03)"
        ],
        "validacion": "¿La OT quedó preparada en IW32 con operaciones, duración/dotación, UT por función, materiales y servicios (PM03) si aplica?",
        "next": "D1_Necesita_recursos"
    },

    "D1_Necesita_recursos": {
        "type": "decision",
        "titulo": "¿Se necesitan recursos?",
        "rol": "Supervisor de Mantenimiento",
        "descripcion": "Decidir si para esta OT se requiere gestionar recursos (materiales/servicios) antes de continuar.",
        "pregunta": "¿Se necesitan recursos?",
        "store_key": "necesita_recursos",
        "opciones": [
            {"label": "SÍ", "value": "SI", "next": "D2_Modo_escoges"},
            {"label": "NO", "value": "NO", "next": "T4_Liberar_OT"}
        ],
        "ayuda": "Si la respuesta es SÍ, debe escoger modo (PEP o CECO) antes de liberar la OT."
    },

    "D2_Modo_escoges": {
        "type": "decision",
        "titulo": "¿Qué modo escoges?",
        "rol": "Supervisor de Mantenimiento",
        "descripcion": "Seleccione el modo de imputación para la OT: CECO o PEP.",
        "pregunta": "¿Qué modo escoges?",
        "store_key": "modo_imputacion",
        "opciones": [
            {"label": "CECO", "value": "CECO", "next": "T3b_Asignar_CECO"},
            {"label": "PEP", "value": "PEP", "next": "T3_Asignar_PEP"}
        ],
        "ayuda": "Seleccione CECO o PEP. En PEP se debe elegir tipo y registrar número. En CECO se debe registrar número de CECO."
    },

    "T3_Asignar_PEP": {
        "type": "task",
        "titulo": "Asignar PEP (S/A)",
        "rol": "Supervisor de Mantenimiento",
        "descripcion": (
            "Si se determinó que aplica PEP, debe indicar cuál corresponde (Mantenimiento mayor / Caso base) y registrar el N° de PEP en la OT.\n"
            "Transacción: IW32"
        ),
        "acciones": [
            "Seleccionar qué aplica: Mantenimiento mayor o Caso base.",
            "Ingresar el N° de PEP.",
            "Asignar el código del elemento PEP en IW32 (pestaña Datos Adic.)."
        ],
        "checklist": [
            "PEP asignado en IW32 (Datos Adic.)"
        ],
        "validacion": "¿Quedó seleccionado el tipo (Mantenimiento mayor o Caso base), registrado el N° de PEP y asignado el PEP en IW32 (Datos Adic.)?",
        "next": "T4_Liberar_OT"
    },

    "T3b_Asignar_CECO": {
        "type": "task",
        "titulo": "Asignar CECO",
        "rol": "Supervisor de Mantenimiento",
        "descripcion": (
            "Asignar el CECO en la OT cuando se requiere imputación por Centro de Costo (CECO).\n"
            "Aplica: Mantenimiento cotidiano.\n"
            "Transacción: IW32"
        ),
        "acciones": [
            "Registrar el número de CECO en la OT.",
            "Confirmar que aplica a Mantenimiento cotidiano."
        ],
        "checklist": [
            "CECO asignado"
        ],
        "validacion": "¿El CECO quedó asignado en la OT (IW32) para Mantenimiento cotidiano?",
        "next": "T4_Liberar_OT"
    },


    "T4_Liberar_OT": {
        "type": "task",
        "titulo": "Liberar OT",
        "rol": "Supervisor de Mantenimiento",
        "descripcion": (
            "Una vez creada y Preparada la OT, pasa por una revisión y confirmación de los trabajos requeridos por parte de la supervisión de "
            "mantenimiento llamada \"Liberar OT\".\n"
            "Transacción: IW32\n"
            "Consideración: En caso de que se evidencie previo a la liberación de una OT que esta no corresponde para su ejecución, "
            "se debe asignar el estatus NEJE a nivel de cabecera de la OT."
        ),
        "acciones": [
            "Revisar y confirmar trabajos requeridos.",
            "Liberar OT en IW32.",
            "Si NO corresponde ejecutar, asignar estatus NEJE (cabecera) antes de liberar."
        ],
        "checklist": [
            "OT revisada por supervisión",
            "OT liberada en IW32 o estatus NEJE aplicado (si no corresponde ejecutar)"
        ],
        "validacion": "¿La OT fue revisada y liberada correctamente (o se aplicó estatus NEJE si no corresponde ejecutar)?",
        "next": "T5_Preparar_trabajos"
    },

    # ---- Preparación / programación / ejecución ----
    "T5_Preparar_trabajos": {
        "type": "task",
        "titulo": "Preparar trabajos",
        "rol": "Especialista/ Ingeniero de Mantenimiento",
        "descripcion": (
            "Asegurar que tengo todo lo necesario para ejecutar los trabajos requeridos.\n"
            "Transacción: No aplica"
        ),
        "acciones": [
            "Verificar procedimientos aplicables.",
            "Verificar licitación/servicios si aplica.",
            "Verificar Permiso de Trabajo (PT) considerado para la ejecución.",
            "Revisar informes anteriores si existen."
        ],
        "checklist": [
            "Procedimientos disponibles",
            "Servicios/licitación verificados (si aplica)",
            "PT considerado (previo a ejecución)",
            "Informes anteriores revisados (si aplica)"
        ],
        "validacion": "¿Está todo lo necesario para ejecutar (procedimientos, servicios si aplica, PT e informes anteriores si aplica)?",
        "next": "R1_Ruta_post_preparar_trabajos"
    },

    "R1_Ruta_post_preparar_trabajos": {
        "type": "router",
        "titulo": "Ruta automática (post Preparar trabajos)",
        "rol": "HMI (auto)",
        "descripcion": "Enrutamiento automático según la decisión '¿Se necesitan recursos?'.",
        "route_on": "necesita_recursos",
        "map": {
            "SI": "T6_Gestion_de_recursos",
            "NO": "T8_Programacion_de_trabajos"
        }
    },

    "T6_Gestion_de_recursos": {
        "type": "task",
        "titulo": "Gestión de recursos",
        "rol": "Abastecimiento y Compras",
        "descripcion": "Gestión de recursos según flujo PRO174 (materiales/servicios).",
        "acciones": ["Gestionar recursos requeridos (materiales/servicios) para habilitar ejecución."],
        "checklist": ["Recursos gestionados/confirmados"],
        "validacion": "¿Los recursos necesarios fueron gestionados/confirmados para la ejecución?",
        "next": "T8_Programacion_de_trabajos"
    },

    "T7_Recepcion_materiales_servicios": {
        "type": "task",
        "titulo": "Recepción de materiales/servicios",
        "rol": "Abastecimiento y Compras",
        "descripcion": "Recepción de materiales/servicios según flujo PRO174.",
        "acciones": ["Confirmar recepción y disponibilidad de materiales/servicios para ejecución."],
        "checklist": ["Materiales recepcionados (si aplica)", "Servicios disponibles/confirmados (si aplica)"],
        "validacion": "¿Materiales/servicios están recepcionados y disponibles para ejecutar?",
        "next": "T8_Programacion_de_trabajos"
    },

    "T8_Programacion_de_trabajos": {
        "type": "task",
        "titulo": "Programación de trabajos",
        "rol": "Planificación / Programación",
        "descripcion": "Programación de trabajos según flujo PRO174.",
        "acciones": ["Programar ventana/fecha de ejecución y coordinación operativa."],
        "checklist": ["Trabajo programado", "Coordinación comunicada"],
        "validacion": "¿El trabajo quedó programado y coordinado para su ejecución?",
        "next": "T9_PT_en_vigencia"
    },

    "T9_PT_en_vigencia": {
        "type": "task",
        "titulo": "PT en estado “En vigencia”",
        "rol": "Planificación / Programación",
        "descripcion": "Condición habilitante para ejecución: PT en estado “En vigencia” (según flujo PRO174).",
        "acciones": ["Verificar que el Permiso de Trabajo (PT) se encuentra vigente antes de ejecutar."],
        "checklist": ["PT vigente confirmado"],
        "validacion": "¿El PT está en estado “En vigencia” y cubre el alcance real del trabajo?",
        "next": "R2_Ruta_post_PT"
    },

    "R2_Ruta_post_PT": {
        "type": "router",
        "titulo": "Ruta automática (post PT vigente)",
        "rol": "HMI (auto)",
        "descripcion": "Si se necesitan recursos, continúa a Retirar materiales; si no, converge directo a Ejecutar trabajos.",
        "route_on": "necesita_recursos",
        "map": {
            "SI": "T10_Retirar_materiales",
            "NO": "T11_Ejecutar_trabajos"
        }
    },


    "T10_Retirar_materiales": {
        "type": "task",
        "titulo": "Retirar materiales",
        "rol": "Técnico de Mantenimiento",
        "descripcion": (
            "El mantenedor procederá con el retiro de materiales de bodega/almacén y solicitará la aprobación de servicios externos contratados si aplica.\n"
            "Transacción: No aplica"
        ),
        "acciones": [
            "Retirar materiales desde bodega/almacén.",
            "Solicitar aprobación de servicios externos contratados si aplica."
        ],
        "checklist": [
            "Materiales retirados",
            "Si aplica: aprobación de servicios externos gestionada"
        ],
        "validacion": "¿Materiales retirados y, si aplica, servicios externos aprobados para ejecutar?",
        "next": "T11_Ejecutar_trabajos"
    },

    "T11_Ejecutar_trabajos": {
        "type": "task",
        "titulo": "Ejecutar trabajos",
        "rol": "Técnico de Mantenimiento",
        "descripcion": (
            "El mantenedor procederá con la ejecución de las tareas descritas como operaciones en la Orden de Trabajo.\n"
            "Transacción: No aplica"
        ),
        "acciones": ["Ejecutar las operaciones de la OT en terreno, bajo PT vigente y estándares aplicables."],
        "checklist": ["Operaciones ejecutadas en terreno", "Condición segura mantenida durante ejecución"],
        "validacion": "¿Se ejecutaron las operaciones de la OT en terreno bajo condiciones seguras y PT vigente?",
        "next": "T12_Notificar_trabajos"
    },

    "T12_Notificar_trabajos": {
        "type": "task",
        "titulo": "Notificar trabajos",
        "rol": "Técnico de Mantenimiento",
        "descripcion": (
            "Una vez concluidos los trabajos en terreno, los mantenedores que participaron del trabajo realizado, deben registrar sus tiempos por "
            "operación asignada y para el caso de no haber realizado alguna tarea se debe dejar por escrito el motivo de la desviación.\n"
            "Transacciones: IW41 / IW48"
        ),
        "acciones": [
            "Notificar trabajos en IW41 o IW48.",
            "Registrar tiempos por operación asignada.",
            "Si no se realizó alguna tarea, dejar por escrito el motivo de la desviación."
        ],
        "checklist": [
            "Notificación registrada (IW41/IW48)",
            "Tiempos por operación registrados",
            "Si aplica: desviaciones justificadas por escrito"
        ],
        "validacion": "¿Se notificaron los trabajos con tiempos por operación y (si aplica) desviaciones justificadas por escrito?",
        "next": "D2_Informe_requerido"
    },

    # ---- Informe ----
    "D2_Informe_requerido": {
        "type": "decision",
        "titulo": "¿Se requiere un informe de los trabajos realizados?",
        "rol": "Supervisor de Mantenimiento",
        "descripcion": "Decisión del supervisor de mantenimiento según PRO174.",
        "pregunta": "¿Se requiere un informe de los trabajos realizados?",
        "opciones": [
            {"label": "SÍ", "next": "T13_Realizar_informe"},
            {"label": "NO", "next": "T16_Revisar_OT"},
        ],
    },

    "T13_Realizar_informe": {
        "type": "task",
        "titulo": "Realizar informe de trabajos realizados",
        "rol": "Mantenimiento",
        "descripcion": (
            "Es decisión del supervisor de mantenimiento definir si los trabajos realizados, requieren un informe. Algunas de las situaciones donde se puede dar son:\n"
            "▪ La especificidad de los trabajos amerita documentar y evidenciar lo realizado.\n"
            "▪ Actividades en las cuales se realizan mediciones que se deben documentar.\n"
            "▪ Trabajos con servicios externos, entre otros.\n"
            "Este informe debe ser realizado por el ejecutor de los trabajos y debe incluir los detalles de lo realizado.\n"
            "Transacción: No aplica"
        ),
        "acciones": [
            "Realizar informe de trabajos realizados con detalle de lo ejecutado.",
            "Incluir mediciones/evidencia cuando corresponda.",
            "Incluir participación de terceros/servicios cuando aplique."
        ],
        "checklist": [
            "Informe elaborado por ejecutor",
            "Detalle de lo realizado incluido",
            "Si aplica: mediciones/evidencia incorporadas",
            "Si aplica: participación de terceros documentada"
        ],
        "validacion": "¿El informe fue realizado por el ejecutor e incluye el detalle de lo realizado (y mediciones/evidencia si aplica)?",
        "next": "T14_Revisar_informe_y_cargar"
    },

    "T14_Revisar_informe_y_cargar": {
        "type": "task",
        "titulo": "Revisar informe de trabajos realizados y cargar en SAP",
        "rol": "Supervisor de Mantenimiento",
        "descripcion": (
            "El supervisor de mantenimiento debe revisar el informe realizado y evaluar si este se encuentra completo.\n"
            "Luego debe almacenar el informe en el gestor documental de la compañía (Sharepoint).\n"
            "Transacción: No aplica"
        ),
        "acciones": [
            "Revisar informe realizado.",
            "Evaluar completitud del informe.",
            "Almacenar informe en Sharepoint (gestor documental)."
        ],
        "checklist": [
            "Informe revisado por supervisor",
            "Informe almacenado en Sharepoint"
        ],
        "validacion": "¿El supervisor revisó el informe y quedó almacenado en Sharepoint?",
        "next": "D3_Informe_completo"
    },

    "D3_Informe_completo": {
        "type": "decision",
        "titulo": "¿Informe completo?",
        "rol": "Supervisor de Mantenimiento",
        "descripcion": "Rombo del flujo PRO174 para validar completitud del informe.",
        "pregunta": "¿Informe completo?",
        "opciones": [
            {"label": "SÍ", "next": "T16_Revisar_OT"},
            {"label": "NO", "next": "T13_Realizar_informe"},
        ],
    },

    # ---- Cierre / Hallazgos ----
    "T16_Revisar_OT": {
        "type": "task",
        "titulo": "Revisar OT",
        "rol": "Supervisor de Mantenimiento",
        "descripcion": (
            "El supervisor revisará desde la OT que los trabajos fueron ejecutados y notificados, conforme a los recursos y tiempos planificados.\n"
            "Transacciones: IW32 / IW33"
        ),
        "acciones": ["Revisar en IW32/IW33 consistencia entre ejecución/notificación y plan (recursos/tiempos)."],
        "checklist": ["OT revisada en IW32/IW33", "Consistencia ejecución/notificación vs plan confirmada"],
        "validacion": "¿La OT fue revisada en IW32/IW33 y es consistente con recursos/tiempos planificados?",
        "next": "T17_Cerrar_OT_CTEC"
    },

    "T17_Cerrar_OT_CTEC": {
        "type": "task",
        "titulo": "Cerrar OT (CTEC)",
        "rol": "Supervisor de Mantenimiento",
        "descripcion": "Una vez concluidos los trabajos de mantenimiento y notificados en el sistema, el supervisor procede a cerrar técnicamente la OT. Transacción: IW32.",
        "acciones": ["Cerrar técnicamente la OT (CTEC) en IW32."],
        "checklist": ["OT cerrada técnicamente (CTEC)"],
        "validacion": "¿La OT fue cerrada técnicamente (CTEC) en IW32?",
        "next": "D4_Hallazgos"
    },

    "D4_Hallazgos": {
        "type": "decision",
        "titulo": "¿Se detectaron hallazgos?",
        "rol": "Mantenimiento",
        "descripcion": "Rombo final del flujo PRO174. Si se detectan hallazgos, se gestiona aviso A3 según PRO071.",
        "pregunta": "¿Se detectaron hallazgos?",
        "opciones": [
            {"label": "SÍ", "next": "T18_Gestionar_aviso_A3"},
            {"label": "NO", "next": "T19_Subir_archivos_pertinentes"},
        ],
    },

    "T18_Gestionar_aviso_A3": {
        "type": "task",
        "titulo": "Gestionar aviso A3 (PRO071)",
        "rol": "Especialista/ Ingeniero de Mantenimiento",
        "descripcion": (
            "En caso de que se detecten hallazgos producto de la ejecución de mantenimientos preventivos o predictivos, "
            "estos se deben gestionar según PRO071 – Gestión de Avisos (Aviso A3).\n"
            "Transacciones: IW21 / IW22"
        ),
        "acciones": [
            "Crear/actualizar aviso A3 según PRO071 (IW21/IW22).",
            "Registrar hallazgo con detalle suficiente para seguimiento."
        ],
        "checklist": ["Aviso A3 gestionado (IW21/IW22)", "Hallazgo registrado con detalle suficiente"],
        "validacion": "¿El aviso A3 quedó gestionado según PRO071 y el hallazgo quedó registrado con detalle suficiente?",
        "next": "T19_Subir_archivos_pertinentes"
    },


    "T19_Subir_archivos_pertinentes": {
        "type": "task",
        "titulo": "Opcional: Subir archivos pertinentes",
        "rol": "HMI",
        "descripcion": "Paso opcional para registrar enlaces/rutas/nombres de archivos pertinentes al mantenimiento (por ejemplo: informes, fotos, evidencias, respaldos).",
        "acciones": [
            "Opcional: pegue aquí enlaces, rutas o nombres de archivos pertinentes (SharePoint, carpeta, etc.)."
        ],
        "checklist": [],
        "validacion": "",
        "next": "END_FIN"
    },

"END_FIN": {
        "type": "end",
        "titulo": "🏁 Fin de mantenimiento",
        "rol": "HMI",
        "descripcion": "Se completaron los pasos del PRO174 (Mantenimiento Preventivo). Puede exportar el JSON auditable si lo requiere.",
        "mensaje": "Ya está en el final/cierre del procedimiento.",
        "estado_final": FINALIZADO
    }
}

# -------------------------
# HMI (estilo visual BASE OFICIAL aprobado)
# -------------------------
class PRO174HMI:
    def __init__(self):
        self.nodo_id = "S0_inicio"
        self.historial = []
        self.logs = []
        self.decisiones = []
        self.bloqueos = []

        self.form_data = {}

        self.run_id = str(uuid.uuid4())
        self.estado = EN_CURSO
        self.start_ts = _now_iso()
        self.end_ts = None

        self.output = widgets.Output(layout={"width":"100%"})

        self.btn_si = widgets.Button(description="SÍ", button_style="success", layout={"width":"48%","height":"44px"})
        self.btn_no = widgets.Button(description="NO", button_style="danger", layout={"width":"48%","height":"44px"})
        self.btn_volver = widgets.Button(description="⬅ Volver al paso anterior", layout={"width":"100%","height":"40px"})
        self.btn_exportar = widgets.Button(description="Exportar JSON (trazabilidad)", icon="download", layout={"width":"100%","height":"40px"})

        self.msg_box = widgets.HTML("")
        self.main_box = widgets.VBox([])

        self.is_blocked = False
        self.block_panel = widgets.VBox([])
        self.btn_rehacer = widgets.Button(description="🔄 Rehacer paso", button_style="info", layout={"width":"100%","height":"40px"})
        self.btn_rehacer.on_click(self._on_rehacer)

        self._decision_widget = None
        self._check_widgets = []

        self._wire()
        self._render()

    def _wire(self):
        self.btn_si.on_click(self._on_si)
        self.btn_no.on_click(self._on_no)
        self.btn_volver.on_click(self._on_volver)
        self.btn_exportar.on_click(self._on_exportar)

    def _log(self, tipo, data=None):
        self.logs.append({
            "ts": _now_iso(),
            "tipo": tipo,
            "estado": self.estado,
            "nodo": self.nodo_id,
            "data": data or {}
        })

    def _push_hist(self):
        self.historial.append(self.nodo_id)

    def _pop_hist(self):
        if self.historial:
            return self.historial.pop()
        return None

    def _set_msg(self, html):
        self.msg_box.value = html

    def _clear_msg(self):
        self.msg_box.value = ""

    def _render_header(self, n):
        badge = f"<span style='display:inline-block;padding:4px 10px;border-radius:999px;background:#eef2ff;border:1px solid #c7d2fe;font-size:12px;color:black;'><b>ROL:</b> {n.get('rol','')}</span>"
        return widgets.HTML(f"""
        <div style="padding:16px;border-radius:12px;background:#f8fafc;border:1px solid #e2e8f0;">
            <div style="font-size:12px;color:#0f172a;"><b>PRO174</b> – Mantenimiento Preventivo</div>
            <div style="margin-top:6px;font-size:22px;color:#0f172a;"><b>{n.get('titulo','')}</b></div>
            <div style="margin-top:8px;">{badge}</div>
            <div style="margin-top:10px;color:#0f172a;font-size:14px;line-height:1.35;white-space:pre-wrap;">{n.get('descripcion','')}</div>
        </div>
        """)

    def _render_task(self, n):
        acciones = "".join([f"<li style='margin:4px 0;color:#0f172a;'>{a}</li>" for a in n.get("acciones",[])])
        valid = n.get("validacion","")

        self._check_widgets = [
            widgets.Checkbox(
                description=item,
                value=False,
                layout=widgets.Layout(width="100%"),
                style={"description_width":"initial"}
            )
            for item in (n.get("checklist",[]) or [])
        ]

        accion_box = widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>⚙️ ACCIÓN A EJECUTAR (texto PRO174)</b></div>
                <ul style="margin-top:10px;padding-left:18px;color:#0f172a;">{acciones}</ul>
            </div>
        """)

        # -------------------------
        # Campos obligatorios (HMI)
        # -------------------------
        extra_sections = []

        # Preparar OT: ingresar N° OT (obligatorio)
        if self.nodo_id == "T2_Preparar_OT":
            self.txt_num_ot = widgets.Text(
                placeholder="Ingrese número de OT (obligatorio)",
                value=str(self.form_data.get("numero_ot","") or ""),
                layout=widgets.Layout(width="100%")
            )
            def _on_ot_change(change):
                self.form_data["numero_ot"] = (change["new"] or "").strip()
            self.txt_num_ot.observe(_on_ot_change, names="value")

            extra_sections.append(widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>🧾 DATOS OBLIGATORIOS</b></div>
                    <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.9;">Debe ingresar el número de OT para poder avanzar.</div>
                </div>
                """),
                self.txt_num_ot
            ]))

        # Preparar trabajos: ingresar N° PT (obligatorio)
        if self.nodo_id == "T5_Preparar_trabajos":
            self.txt_num_pt = widgets.Text(
                placeholder="Ingrese número de PT (obligatorio)",
                value=str(self.form_data.get("numero_pt","") or ""),
                layout=widgets.Layout(width="100%")
            )
            def _on_pt_change(change):
                self.form_data["numero_pt"] = (change["new"] or "").strip()
            self.txt_num_pt.observe(_on_pt_change, names="value")

            extra_sections.append(widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>🧾 DATOS OBLIGATORIOS</b></div>
                    <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.9;">Debe ingresar el número de PT para poder avanzar.</div>
                </div>
                """),
                self.txt_num_pt
            ]))

        # Asignar PEP: selección excluyente + ingresar N° PEP (obligatorio)

        # Subir archivos pertinentes (opcional)
        if self.nodo_id == "T19_Subir_archivos_pertinentes":
            self.txt_archivos = widgets.Textarea(
                placeholder="Opcional: sube los archivos pertinentes (pegue enlaces/rutas/nombres aquí)",
                value=str(self.form_data.get("archivos_pertinentes","") or ""),
                layout=widgets.Layout(width="100%", height="110px")
            )
            def _on_arch_change(change):
                self.form_data["archivos_pertinentes"] = (change["new"] or "").strip()
            self.txt_archivos.observe(_on_arch_change, names="value")

            extra_sections.append(widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>📎 ADJUNTOS (OPCIONAL)</b></div>
                    <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.9;">Si corresponde, registre aquí enlaces/rutas/nombres de archivos (SharePoint, carpeta, etc.).</div>
                </div>
                """),
                self.txt_archivos
            ]))
        if self.nodo_id == "T3_Asignar_PEP":
            self.chk_pep_mayor = widgets.Checkbox(
                description="Mantenimiento mayor",
                value=(self.form_data.get("pep_tipo") == "Mantenimiento mayor"),
                layout=widgets.Layout(width="100%"),
                style={"description_width":"initial"}
            )
            self.chk_pep_caso_base = widgets.Checkbox(
                description="Caso base",
                value=(self.form_data.get("pep_tipo") == "Caso base"),
                layout=widgets.Layout(width="100%"),
                style={"description_width":"initial"}
            )

            self.txt_num_pep = widgets.Text(
                placeholder="Ingrese número de PEP (obligatorio)",
                value=str(self.form_data.get("numero_pep","") or ""),
                layout=widgets.Layout(width="100%")
            )

            # Estado inicial del input PEP
            pep_selected = bool(self.chk_pep_mayor.value or self.chk_pep_caso_base.value)
            self.txt_num_pep.disabled = (not pep_selected)

            def _set_pep_tipo(tipo):
                self.form_data["pep_tipo"] = tipo
                self.txt_num_pep.disabled = False

            def _on_mayor_change(change):
                if change["new"]:
                    # Exclusión
                    self.chk_pep_caso_base.value = False
                    _set_pep_tipo("Mantenimiento mayor")
                else:
                    if not self.chk_pep_caso_base.value:
                        self.form_data["pep_tipo"] = ""

            def _on_caso_change(change):
                if change["new"]:
                    self.chk_pep_mayor.value = False
                    _set_pep_tipo("Caso base")
                else:
                    if not self.chk_pep_mayor.value:
                        self.form_data["pep_tipo"] = ""

            def _on_pep_change(change):
                self.form_data["numero_pep"] = (change["new"] or "").strip()

            self.chk_pep_mayor.observe(_on_mayor_change, names="value")
            self.chk_pep_caso_base.observe(_on_caso_change, names="value")
            self.txt_num_pep.observe(_on_pep_change, names="value")

            extra_sections.append(widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>📌 ¿CUÁL APLICA?</b></div>
                    <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.9;">Seleccione solo una opción para poder avanzar.</div>
                </div>
                """),
                widgets.VBox([self.chk_pep_mayor, self.chk_pep_caso_base]),
                widgets.HTML("<div style='height:8px;'></div>"),
                self.txt_num_pep
            ]))


        # Asignar CECO: ingresar N° CECO (obligatorio)
        if self.nodo_id == "T3b_Asignar_CECO":
            # Aplica fijo: Mantenimiento cotidiano
            self.form_data["ceco_aplica"] = "Mantenimiento cotidiano"
            self.txt_num_ceco = widgets.Text(
                placeholder="Ingrese número de CECO (obligatorio)",
                value=str(self.form_data.get("numero_ceco","") or ""),
                layout=widgets.Layout(width="100%")
            )
            def _on_ceco_change(change):
                self.form_data["numero_ceco"] = (change["new"] or "").strip()
            self.txt_num_ceco.observe(_on_ceco_change, names="value")

            extra_sections.append(widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>🧾 DATOS OBLIGATORIOS</b></div>
                    <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.9;">Aplica: <b>Mantenimiento cotidiano</b>. Debe ingresar el número de CECO para poder avanzar.</div>
                </div>
                """),
                self.txt_num_ceco
            ]))


        extra_box = widgets.VBox(extra_sections) if extra_sections else widgets.VBox([])

        checklist_box = widgets.VBox([])
        if self._check_widgets:
            checklist_box = widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>🧾 CHECKLIST (obligatorio para avanzar)</b></div>
                    <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.9;">Marca cada ítem al completar en terreno.</div>
                </div>
                """),
                widgets.VBox(self._check_widgets)
            ])

        valid_box = widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #0ea5e9;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>✅ ¡VALIDACIÓN!</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{valid}</b></div>
                <div style="margin-top:6px;font-size:12px;color:#0f172a;">Confirma con <b>SÍ</b> para avanzar. Si respondes <b>NO</b>, el paso queda bloqueado.</div>
            </div>
        """)

        return widgets.VBox([accion_box, extra_box, checklist_box, valid_box])

    def _render_decision(self, n):
        opts = n.get("opciones",[])
        radios = widgets.RadioButtons(
            options=[(o["label"], o["next"]) for o in opts],
            layout={"width":"100%"},
            style={"description_width":"initial"},
        )
        help_txt = n.get("ayuda","")
        help_html = f"<div style='margin-top:10px;font-size:12px;color:#0f172a;opacity:0.9;'><b>Nota:</b> {help_txt}</div>" if help_txt else ""
        return widgets.VBox([
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>🔶 DECISIÓN (rombo)</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{n.get('pregunta','')}</b></div>
                {help_html}
            </div>
            """),
            radios
        ]), radios

    def _render_block_panel(self):
        if not self.is_blocked:
            self.block_panel.children = []
            return

        title = widgets.HTML("""
        <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #ef4444;background:#fff1f2;">
            <div style="font-size:14px;color:#0f172a;"><b>⛔ BLOQUEADO</b> — Seleccione motivo(s) y registre detalle.</div>
            <div style="margin-top:8px;font-size:12px;color:#0f172a;">No puede avanzar hasta rehacer el paso.</div>
        </div>
        """)

        self.sel_motivos = widgets.SelectMultiple(options=MOTIVOS_BLOQUEO_PRO174, rows=8, layout={"width":"100%"})
        self.txt_detalle = widgets.Textarea(
            placeholder="Detalle del bloqueo (obligatorio si selecciona 'Otro').",
            layout=widgets.Layout(width="100%", height="80px")
        )

        self.block_panel.children = [
            title,
            widgets.HTML("<b>Motivo(s) de bloqueo:</b> (selección múltiple)"),
            self.sel_motivos,
            widgets.HTML("<b>Detalle:</b>"),
            self.txt_detalle,
            self.btn_rehacer
        ]

    def _render_footer(self):
        self.btn_volver.disabled = (len(self.historial) == 0)
        return widgets.VBox([
            widgets.HBox([self.btn_si, self.btn_no], layout={"justify_content":"space-between","margin":"10px 0"}),
            self.btn_volver,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.btn_exportar,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.block_panel,
            self.msg_box,
        ])

    def _render(self):
        with self.output:
            self.output.clear_output()
            self._clear_msg()

            n = NODOS[self.nodo_id]
            css = widgets.HTML("""
            <style>
              .widget-label { white-space: normal !important; }
              .jp-InputArea .widget-label { white-space: normal !important; }
              .jupyter-widgets label { white-space: normal !important; }
            </style>
            """)
            header = self._render_header(n)

            if n["type"] == "task":
                body = self._render_task(n)
                self._decision_widget = None
            elif n["type"] == "decision":
                body, radios = self._render_decision(n)
                self._decision_widget = radios
                self._check_widgets = []
            elif n["type"] == "end":
                self._decision_widget = None
                self._check_widgets = []

                esc = lambda x: (str(x) if x is not None else "").replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
                num_ot = esc(self.form_data.get("numero_ot",""))
                num_pt = esc(self.form_data.get("numero_pt",""))
                modo = esc(self.form_data.get("modo_imputacion",""))
                num_pep = esc(self.form_data.get("numero_pep",""))
                num_ceco = esc(self.form_data.get("numero_ceco",""))

                imputacion_label = "PEP" if (modo or "").strip().upper() == "PEP" else ("CECO" if (modo or "").strip().upper() == "CECO" else "PEP/CECO")
                imputacion_val = num_pep if imputacion_label == "PEP" else (num_ceco if imputacion_label == "CECO" else (num_pep or num_ceco))

                resumen_tbl = f"""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                  <div style="font-size:13px;color:#0f172a;"><b>📋 RESUMEN (inputs)</b></div>
                  <table style="width:100%;margin-top:10px;border-collapse:collapse;font-size:13px;color:#0f172a;">
                    <tr>
                      <td style="padding:8px;border-bottom:1px solid #e2e8f0;width:35%;"><b>N° OT</b></td>
                      <td style="padding:8px;border-bottom:1px solid #e2e8f0;"><code>{num_ot or "-"}</code></td>
                    </tr>
                    <tr>
                      <td style="padding:8px;border-bottom:1px solid #e2e8f0;"><b>{imputacion_label}</b></td>
                      <td style="padding:8px;border-bottom:1px solid #e2e8f0;"><code>{imputacion_val or "-"}</code></td>
                    </tr>
                    <tr>
                      <td style="padding:8px;"><b>N° PT</b></td>
                      <td style="padding:8px;"><code>{num_pt or "-"}</code></td>
                    </tr>
                  </table>
                </div>
                """

                # Decisiones (tal como aparecen en el JSON exportado)
                if self.decisiones:
                    dec_rows = ""
                    for d in self.decisiones:
                        dec_rows += f"""
                        <tr>
                          <td style="padding:8px;border-bottom:1px solid #e2e8f0;white-space:nowrap;"><code>{esc(d.get('ts',''))}</code></td>
                          <td style="padding:8px;border-bottom:1px solid #e2e8f0;">{esc(d.get('titulo',''))}</td>
                          <td style="padding:8px;border-bottom:1px solid #e2e8f0;"><b>{esc(d.get('seleccion',''))}</b></td>
                        </tr>
                        """
                    dec_tbl = f"""
                    <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                      <div style="font-size:13px;color:#0f172a;"><b>🔎 DECISIONES (trazabilidad)</b></div>
                      <table style="width:100%;margin-top:10px;border-collapse:collapse;font-size:13px;color:#0f172a;">
                        <thead>
                          <tr>
                            <th style="text-align:left;padding:8px;border-bottom:1px solid #cbd5e1;">TS</th>
                            <th style="text-align:left;padding:8px;border-bottom:1px solid #cbd5e1;">Nodo</th>
                            <th style="text-align:left;padding:8px;border-bottom:1px solid #cbd5e1;">Selección</th>
                          </tr>
                        </thead>
                        <tbody>
                          {dec_rows}
                        </tbody>
                      </table>
                    </div>
                    """
                else:
                    dec_tbl = f"""
                    <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                      <div style="font-size:13px;color:#0f172a;"><b>🔎 DECISIONES (trazabilidad)</b></div>
                      <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.85;">No se registraron decisiones.</div>
                    </div>
                    """

                body = widgets.HTML(f"""
                <div style="margin-top:12px;padding:18px;border-radius:12px;border:2px solid #22c55e;background:#f0fdf4;">
                    <div style="font-size:20px;color:#0f172a;"><b>🏁 FIN</b></div>
                    <div style="margin-top:10px;font-size:15px;color:#0f172a;">{n.get('mensaje','')}</div>
                    <div style="margin-top:10px;font-size:12px;color:#0f172a;">Estado final: <b>{n.get('estado_final','')}</b></div>
                </div>
                {resumen_tbl}
                {dec_tbl}
                """)
            else:
                body = widgets.HTML("<div>Tipo de nodo no soportado.</div>")
                self._decision_widget = None
                self._check_widgets = []

            self._render_block_panel()
            footer = self._render_footer()
            self.main_box.children = [css, header, body, footer]
            display(self.main_box)

    def _check_ready_to_advance(self):
        n = NODOS[self.nodo_id]

        if self.is_blocked:
            return False, "Paso bloqueado. Registre motivo(s) y use 'Rehacer paso'."

        if n["type"] == "end":
            return True, ""

        if n["type"] == "decision":
            if self._decision_widget is None or self._decision_widget.value is None:
                return False, "Debe seleccionar una opción para avanzar."
            return True, ""

        if n["type"] == "router":
            return True, ""

        if n["type"] == "task":
            # Checklist: ítems "Si aplica:" / "(si aplica)" NO son obligatorios para avanzar
            if self._check_widgets:
                # Caso especial: Recepción de materiales/servicios -> basta con marcar 1 o 2 (pero al menos 1)
                if self.nodo_id == "T7_Recepcion_materiales_servicios":
                    if not any(cb.value for cb in self._check_widgets):
                        return False, "Debe marcar al menos un ítem en Recepción de materiales/servicios para avanzar."
                else:
                    obligatorios = []
                    for cb in self._check_widgets:
                        desc = (cb.description or "").strip().lower()
                        # Se considera "si aplica" como NO obligatorio
                        if desc.startswith("si aplica:") or "(si aplica" in desc:
                            continue
                        obligatorios.append(cb)

                    if obligatorios and not all(cb.value for cb in obligatorios):
                        return False, "Debe completar el checklist obligatorio antes de avanzar."

            # Validaciones por paso (campos HMI obligatorios)
            if self.nodo_id == "T2_Preparar_OT":
                num_ot = (self.form_data.get("numero_ot","") or "").strip()
                if not num_ot:
                    return False, "Debe ingresar el número de OT para avanzar."

            if self.nodo_id == "T5_Preparar_trabajos":
                num_pt = (self.form_data.get("numero_pt","") or "").strip()
                if not num_pt:
                    return False, "Debe ingresar el número de PT para avanzar."

            if self.nodo_id == "T3_Asignar_PEP":
                pep_tipo = (self.form_data.get("pep_tipo","") or "").strip()
                if pep_tipo not in ["Mantenimiento mayor", "Caso base"]:
                    return False, "Debe seleccionar solo una opción (Mantenimiento mayor o Caso base) para avanzar."
                num_pep = (self.form_data.get("numero_pep","") or "").strip()
                if not num_pep:
                    return False, "Debe ingresar el número de PEP para avanzar."


            if self.nodo_id == "T3b_Asignar_CECO":
                num_ceco = (self.form_data.get("numero_ceco","") or "").strip()
                if not num_ceco:
                    return False, "Debe ingresar el número de CECO para avanzar."

            return True, ""

        return True, ""

    def _advance_to(self, next_id):
        if next_id not in NODOS:
            self._set_msg(f"""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff7ed;border:1px solid #fdba74;color:#0f172a;'>
                <b>⚠ Error de flujo:</b> el nodo destino no existe: <code>{next_id}</code>
            </div>
            """)
            self._log("ERROR_FLUJO", {"missing_next": next_id})
            return

        self.nodo_id = next_id

        # Enrutamiento automático (router)
        safety = 0
        while self.nodo_id in NODOS and NODOS[self.nodo_id].get("type") == "router":
            safety += 1
            if safety > 10:
                self._set_msg("""
                <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                    <b>⚠ Error de flujo:</b> bucle de enrutamiento detectado.
                </div>
                """)
                self._log("ERROR_ROUTER_LOOP", {"router": self.nodo_id})
                return

            r = NODOS[self.nodo_id]
            route_on = r.get("route_on")
            val = (self.form_data.get(route_on) or "").strip().upper()
            nxt = (r.get("map") or {}).get(val)

            if not nxt:
                self._set_msg(f"""
                <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                    <b>⚠ Error de flujo:</b> no se puede enrutar desde <code>{self.nodo_id}</code>. Falta valor para <code>{route_on}</code>.
                </div>
                """)
                self._log("ERROR_ROUTER", {"router": self.nodo_id, "route_on": route_on, "value": val})
                return

            self._log("ROUTER", {"router": self.nodo_id, "route_on": route_on, "value": val, "next": nxt})
            self.nodo_id = nxt

        self._render()

    def _on_si(self, _):
        ok, msg = self._check_ready_to_advance()
        if not ok:
            self._set_msg(f"""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ {msg}</b>
            </div>
            """)
            self._log("VALIDACION_FALLA", {"mensaje": msg})
            return

        n = NODOS[self.nodo_id]

        if n["type"] == "end":
            self.estado = FINALIZADO
            self.end_ts = _now_iso()
            self._log("FINALIZA")
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
                <b>🏁 Ya está en el final/cierre.</b>
            </div>
            """)
            return

        self._push_hist()

        if n["type"] == "task":
            self._log("AVANZA", {"next": n.get("next")})
            self._advance_to(n.get("next"))
        elif n["type"] == "decision":
            chosen_next = self._decision_widget.value
            chosen_label = next((o["label"] for o in n.get("opciones",[]) if o["next"] == chosen_next), None)

            # Persistencia de decisión en inputs (si aplica)
            store_key = n.get("store_key")
            if store_key:
                chosen_value = next((o.get("value") for o in n.get("opciones",[]) if o.get("next") == chosen_next), None)
                self.form_data[store_key] = chosen_value if chosen_value is not None else (chosen_label or "")

            self.decisiones.append({
                "ts": _now_iso(),
                "nodo": self.nodo_id,
                "titulo": n.get("titulo",""),
                "seleccion": chosen_label,
                "next": chosen_next
            })
            self._log("DECISION", {"seleccion": chosen_label, "next": chosen_next})
            self._advance_to(chosen_next)

    def _on_no(self, _):
        if self.is_blocked:
            return
        self.is_blocked = True
        self.estado = BLOQUEADO
        self.block_ts_inicio = _now_iso()
        self._log("BLOQUEADO_INICIO")
        self._render()

    def _on_rehacer(self, _):
        motivos = list(self.sel_motivos.value) if hasattr(self, "sel_motivos") else []
        detalle = (self.txt_detalle.value or "").strip() if hasattr(self, "txt_detalle") else ""

        if not motivos:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ Debe seleccionar al menos un motivo.</b>
            </div>
            """)
            self._log("BLOQUEADO_VALIDACION_FALLA", {"mensaje": "sin_motivo"})
            return

        if "Otro" in motivos and not detalle:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ Debe ingresar detalle si selecciona 'Otro'.</b>
            </div>
            """)
            self._log("BLOQUEADO_VALIDACION_FALLA", {"mensaje": "otro_sin_detalle"})
            return

        bloqueo = {
            "ts_inicio": getattr(self, "block_ts_inicio", None),
            "ts_fin": _now_iso(),
            "nodo": self.nodo_id,
            "titulo": NODOS[self.nodo_id].get("titulo",""),
            "motivos": motivos,
            "detalle": detalle
        }
        self.bloqueos.append(bloqueo)
        self._log("BLOQUEADO_FIN", bloqueo)

        self.is_blocked = False
        self.estado = EN_CURSO
        self._log("REHACER_PASO")
        self._render()

    def _on_volver(self, _):
        if self.is_blocked:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ No puede volver mientras el paso está bloqueado. Use 'Rehacer paso'.</b>
            </div>
            """)
            return

        prev_id = self._pop_hist()
        if prev_id is not None:
            self._log("VOLVER", {"to": prev_id})
            self._advance_to(prev_id)

    def _on_exportar(self, _):
        payload = {
            "proceso": "PRO174 – Mantenimiento Preventivo",
            "run_id": self.run_id,
            "estado": self.estado,
            "start_ts": self.start_ts,
            "end_ts": self.end_ts,
            "current_node": self.nodo_id,
            "history_stack": list(self.historial),
            "decisiones": list(self.decisiones),
            "bloqueos": list(self.bloqueos),
            "logs": list(self.logs),
            "inputs": dict(self.form_data),
            "export_ts": _now_iso(),
        }
        pretty = json.dumps(payload, ensure_ascii=False, indent=2)
        self._set_msg(f"""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
            <b>📦 Export JSON (trazabilidad)</b>
            <pre style='white-space:pre-wrap;margin-top:10px;color:#0f172a;'>{pretty}</pre>
        </div>
        """)

    def iniciar(self):
        display(self.output)

hmi = PRO174HMI()
hmi.iniciar()


Output(layout=Layout(width='100%'))